# Two-Stage XGBoost Pipeline — Training

**Feature extraction:** The frozen MobileNetV2 backbone (loaded from `structure_model.keras`) is used as a feature extractor. Each image is passed through the backbone and the 1280-dim `GlobalAveragePooling2D` output is saved as a feature vector. XGBoost models are then trained on these embeddings — XGBoost cannot operate on raw pixels directly.

Stage 1: XGBoost structure classifier (Decks / Pavements / Walls)  
Stage 2: XGBoost binary defect classifier per structure (recall-optimised, same threshold logic as CNN pipeline)

Splits reused from `train_split.csv`, `val_split.csv`, `test_split.csv` — identical to the CNN pipeline for a fair comparison.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
import json
import pickle
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.metrics import recall_score, f1_score

IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
SEED           = 42
STRUCTURES     = ['Decks', 'Pavements', 'Walls']
THRESHOLD_SWEEP = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
MIN_F1         = 0.60
AUTOTUNE       = tf.data.AUTOTUNE

np.random.seed(SEED)
print('XGBoost:', xgb.__version__)
print('TensorFlow:', tf.__version__)

XGBoost: 3.2.0
TensorFlow: 2.16.2


## 1. Load Split CSVs

In [ ]:
df_train = pd.read_csv('splits/train_split.csv')
df_val   = pd.read_csv('splits/val_split.csv')
df_test  = pd.read_csv('splits/test_split.csv')
print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
print()
print(df_train.groupby(['structure', 'label']).size().unstack(fill_value=0))

## 2. Build Feature Extractor

Load the trained `structure_model.keras` and extract the `GlobalAveragePooling2D` layer output (1280-dim) as a sub-model. The backbone weights are **frozen** — no fine-tuning here.

In [ ]:
full_model = tf.keras.models.load_model('models/cnn/structure_model.keras')

# Find GlobalAveragePooling2D layer and use its output as features
gap_layer = next(
    l for l in full_model.layers
    if isinstance(l, tf.keras.layers.GlobalAveragePooling2D)
)

feature_extractor = tf.keras.Model(
    inputs=full_model.input,
    outputs=gap_layer.output,
)
feature_extractor.trainable = False
print(f'Feature extractor output shape: {feature_extractor.output_shape}')

## 3. Feature Extraction (with disk cache)

Extraction takes ~5 min on Apple M3. After the first run, features are loaded from `.npy` cache files.

In [ ]:
def extract_features(df_rows):
    paths = df_rows['path'].values

    def load_image(path):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
        return img

    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return feature_extractor.predict(ds, verbose=1)


Path('features').mkdir(exist_ok=True)
cache = {
    'train': ('features/features_train.npy', df_train),
    'val':   ('features/features_val.npy',   df_val),
    'test':  ('features/features_test.npy',  df_test),
}

features = {}
for split, (fname, df) in cache.items():
    if Path(fname).exists():
        print(f'Loading {fname} from cache...')
        features[split] = np.load(fname)
    else:
        print(f'Extracting {split} features...')
        features[split] = extract_features(df)
        np.save(fname, features[split])
        print(f'  Saved {fname}')

X_train, X_val, X_test = features['train'], features['val'], features['test']
print(f'\nX_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

## 4. Stage 1 — Structure Classifier (Decks / Pavements / Walls)

XGBoost multiclass classifier on 1280-dim embeddings. `multi:softprob` mirrors the CNN's 3-class softmax.

In [ ]:
y_struct_train = df_train['structure_idx'].values
y_struct_val   = df_val['structure_idx'].values

xgb_struct = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

xgb_struct.fit(
    X_train, y_struct_train,
    eval_set=[(X_val, y_struct_val)],
    verbose=50,
)

val_preds = xgb_struct.predict(X_val)
val_acc   = (val_preds == y_struct_val).mean()
print(f'\nStructure classifier val accuracy: {val_acc:.3f}')

Path('models/xgb').mkdir(parents=True, exist_ok=True)
with open('models/xgb/xgb_structure_model.pkl', 'wb') as f:
    pickle.dump(xgb_struct, f)
print('Saved models/xgb/xgb_structure_model.pkl')

## 5. Stage 2 — Per-Structure Defect Classifiers (Recall-Focused)

Three separate binary XGBoost classifiers, one per structure type.

**Imbalance handling:** `scale_pos_weight = n_neg / n_pos` (XGBoost's equivalent of Keras `class_weight`).  
**Threshold tuning:** same sweep logic as the CNN pipeline — lowest threshold achieving F1 ≥ 0.60 on the val set.

In [ ]:
thresholds = {}

for struct in STRUCTURES:
    print(f'\n{"="*60}')
    print(f'  {struct} — XGBoost Defect Classifier')
    print(f'{"="*60}')

    mask_tr = (df_train['structure'] == struct).values
    mask_vl = (df_val['structure']   == struct).values

    X_tr = X_train[mask_tr]
    y_tr = df_train.loc[mask_tr, 'defect_idx'].values
    X_vl = X_val[mask_vl]
    y_vl = df_val.loc[mask_vl, 'defect_idx'].values

    n_pos = int((y_tr == 1).sum())
    n_neg = int((y_tr == 0).sum())
    spw   = n_neg / n_pos
    print(f'  Train: {n_pos} defect / {n_neg} no_defect  |  scale_pos_weight={spw:.2f}')

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        objective='binary:logistic',
        scale_pos_weight=spw,
        eval_metric='logloss',
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        verbose=50,
    )

    fname = f'models/xgb/xgb_defect_{struct.lower()}.pkl'
    with open(fname, 'wb') as f:
        pickle.dump(model, f)
    print(f'  Saved {fname}')

    # Threshold tuning on val set (same logic as CNN pipeline)
    val_probs = model.predict_proba(X_vl)[:, 1]
    print(f'\n  Threshold sweep (val set):')
    best_thresh = 0.50
    found = False
    for t in THRESHOLD_SWEEP:
        preds = (val_probs >= t).astype(int)
        rec = recall_score(y_vl, preds, zero_division=0)
        f1  = f1_score(y_vl, preds, zero_division=0)
        print(f'    t={t:.2f}  recall={rec:.3f}  f1={f1:.3f}')
        if f1 >= MIN_F1 and not found:
            best_thresh = t
            found = True

    if not found:
        f1s = [f1_score(y_vl, (val_probs >= t).astype(int), zero_division=0)
               for t in THRESHOLD_SWEEP]
        best_thresh = THRESHOLD_SWEEP[int(np.argmax(f1s))]
        print(f'  WARNING: no threshold achieved F1>={MIN_F1}; using best-F1 threshold.')

    thresholds[struct] = best_thresh
    print(f'  Chosen threshold for {struct}: {best_thresh}')

with open('models/xgb/xgb_thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)
print('\nSaved models/xgb/xgb_thresholds.json:', thresholds)

## 6. Summary

In [ ]:
print('=== XGBoost Training Complete ===')
print()
artifacts = [
    'models/xgb/xgb_structure_model.pkl',
    'models/xgb/xgb_defect_decks.pkl',
    'models/xgb/xgb_defect_pavements.pkl',
    'models/xgb/xgb_defect_walls.pkl',
    'models/xgb/xgb_thresholds.json',
    'features/features_train.npy',
    'features/features_val.npy',
    'features/features_test.npy',
]
for fname in artifacts:
    status = 'OK' if Path(fname).exists() else 'MISSING'
    print(f'  [{status}] {fname}')
print()
print('Thresholds:', thresholds)